# Mode masking in the $(k_\perp, k_\parallel)$ plane

Build a mask interactively — wedge cut, rectangular limits, carved-out regions —
and see the spherical power spectrum update, for **one or both** variants.

Nothing re-runs the pipeline: everything downstream of the mask ($X$, `sigma`, the
binning, $\Delta^2(k)$) is recomputed from the stored `pk` and `dpkn`.

**Masks are per grid.** The variants have different $N_E$ (668 with SCF, 760
without), so their $k_\parallel$ axes differ and a mask array built for one cannot
be applied to the other. Share the *specification*, build the array per run —
`msk.for_run` does that.

In [ ]:
BASE = '/scratch3/users/schatterjee/project_MWA/hyperdrive_residual-R5_LF_RA39p7/ps_run_RA_39p7DEG'

# one or both; order sets the legend order
VARIANTS = ['without_SCF', 'with_SCF']

import numpy as np
import matplotlib.pyplot as plt

from myutils.config import load
import myutils.plots as mp
import myutils.masking as msk
import myutils.reference as rf

%matplotlib inline
mp.style()

runs, cfgs = {}, {}
for v in VARIANTS:
    cfgs[v] = load(f'{BASE}/{v}/config.{v}.yaml')
    runs[v] = mp.load_run(cfgs[v].paths.output_dir)

NBIN = int(cfgs[VARIANTS[0]].power_spectrum.NBin)

for v, d in runs.items():
    print(f'{v:<14s} NE={int(d["NE"]):<4d} fac={float(d["fac"]):.3f}  '
          f'k_para {d["kpara"].min():.4f}-{d["kpara"].max():.4f}  '
          f'({d["kpara"].size} values)')

## 1 · The mask specification

One spec, applied to every variant. Edit and re-run everything below.

Start from a **preset** and override what you want — `myutils.masking.PRESETS`:

| preset | what it is |
|---|---|
| `paper` | the published mode selection of **TTGE III** (Sarkar et al. 2026, [arXiv:2604.24144](https://arxiv.org/abs/2604.24144), in `Documentation/`) |
| `paper_box` | the same $(k_\perp, k_\parallel)$ region without the per-$k_\perp$ streak windows — more modes, less hand-tuning |
| `none` | no selection at all |

The paper's selection is three cuts plus a streak mask:

- $k_\perp \le 0.045\ \mathrm{Mpc}^{-1}$ — beyond that the wedge boundary rises above the SCF smoothing scale, so filtering alone cannot clean the long baselines
- $k_\parallel \ge 0.135\ \mathrm{Mpc}^{-1}$ — below this SCF itself removes power, so those modes are avoided rather than trusted. **The number is tied to the 2 MHz smoothing scale, which is exactly what this pipeline uses**, so it carries over unchanged
- $k_\parallel \le 1.399\ \mathrm{Mpc}^{-1}$ — above that the estimate is noise-dominated
- inside that box, per-$k_\perp$ windows mask the contaminated horizontal streaks (their Figure 5)

The wedge itself excludes $k_\parallel \le \mathrm{fac}\cdot k_\perp + \mathrm{buffer}$; the buffer sits above the horizon line because the wedge edge is soft.

In [ ]:
# A preset name, or a dict. Keys next to `preset` override it; None means
# "keep what the preset says".
SPEC = dict(
    preset        = 'paper',
    # --- set any of these to depart from the paper's selection ---
    use_wedge     = None,
    wedge_buffer  = None,     # Mpc^-1 above the horizon line
    kpara_min     = None,
    kpara_max     = None,
    kperp_min     = None,
    kperp_max     = None,
    use_tabulated = None,
)

# extra regions to carve out: (kperp_lo, kperp_hi, kpara_lo, kpara_hi), None = open edge
BOXES = []

print('resolved spec:')
for k, v in msk.resolve_spec(SPEC).items():
    print(f'  {k:<16s} {v}')
print()

masks = {}
for v, d in runs.items():
    m, parts = msk.for_run(d, SPEC, BOXES)
    masks[v] = m
    print(f'=== {v} ===')
    print(msk.describe(parts, d['kper'], d['kpara'], m))
    if 'boxes_removed' in parts:
        print(f"  removed by {len(BOXES)} box(es)   {parts['boxes_removed']:,}")
    print()

## 2 · See what survives

The cylindrical PS with the excluded region greyed out and the wedge drawn. Check
this before trusting a mask — a stray box is obvious here and invisible in numbers.

In [ ]:
for v in VARIANTS:
    fig = mp.mask_overlay(runs[v], masks[v],
                          title=f'{v} — {int(masks[v].sum()):,} modes kept')
    display(fig); plt.close(fig)

## 3 · The power spectrum under this mask

`sigma` should sit near 1. A mask that pulls it far away is admitting modes the
noise model does not describe.

In [ ]:
results = {}
for v in VARIANTS:
    r = msk.recompute(runs[v], masks[v], nbin=NBIN)
    results[v] = r
    print(f'{v:<14s} modes {r["nmodes"]:>7,}   sigma {r["sigma"]:>8.3f}   '
          f'mu {r["mu"]:>7.3f}   bins {len(r["kk"])}')

In [ ]:
for v in VARIANTS:
    r = results[v]
    print(f'=== {v}   (sigma = {r["sigma"]:.3f}) ===')
    print(f"{'k [1/Mpc]':>11}  {'Delta^2':>13}  {'2 sigma':>13}  {'SNR':>8}  {'upper limit':>13}")
    for i in range(len(r['kk'])):
        print(f"{r['kk'][i]:11.4f}  {r['dk2'][i]:13.3e}  {r['dpk2'][i]:13.3e}"
              f"  {r['snr'][i]:8.3f}  {r['ul'][i]:13.3e}")
    print()

In [ ]:
# Published limits to draw alongside. Names from myutils.reference.DATASETS;
# [] for none. ttge3_alpha11 is the like-for-like case — a single pointing centre.
REFERENCE = ['ttge3_alpha11', 'ttge3_case_i']

print(rf.describe('ttge3_alpha11'), '\n')
print(rf.describe('ttge3_case_i'))

In [ ]:
def overlay(res, title, reference=REFERENCE):
    fig, ax = plt.subplots(figsize=(7, 4.8))
    for (name, r), colour in zip(res.items(), mp.PALETTE):
        mag = np.abs(r['dk2'])
        at_zero = r['dpk2'] >= mag
        lower = np.where(at_zero, mag * (1 - 1 / mp.ARROW_DROP), r['dpk2'])
        ax.errorbar(r['kk'], mag, yerr=np.vstack([lower, r['dpk2']]),
                    fmt='D', ms=5, color=colour, mec='#1a1a1a', mew=0.7,
                    elinewidth=1.4, capsize=3, uplims=at_zero,
                    label=f"{name}  ($\\sigma$={r['sigma']:.2f})")

    # neutral ink, distinct marker and dash: a published limit must not be mistaken
    # for one of our runs, and adding one must never recolour a run
    refs = mp.draw_reference(ax, reference)
    span = [r['kk'] for r in res.values()] + \
           [np.array([p['k'] for p in ds['points']]) for ds in refs]

    ax.set_yscale('log')
    mp._log_k_axis(ax, np.concatenate(span))
    ax.set_xlabel(r'$k$  [Mpc$^{-1}$]')
    ax.set_ylabel(r'$|\Delta^2(k)|$  [mK$^2$]')
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=8)
    ax.margins(x=0.08, y=0.12)
    mp._legend_headroom(ax)
    return fig

overlay(results, 'same mask specification, both variants')

## 4 · How much does the mask choice matter?

Sweep the wedge buffer. If an upper limit moves a lot, it is driven by a few modes
near the wedge rather than the bulk of the data.

In [ ]:
V = VARIANTS[-1]
d = runs[V]

sweep = {}
for buf in (0.0, 0.05, 0.1, 0.2):
    m, _ = msk.for_run(d, dict(SPEC, wedge_buffer=buf), BOXES)
    r = msk.recompute(d, m, nbin=NBIN)
    sweep[f'buffer {buf}'] = r
    print(f'buffer {buf:<5} modes {r["nmodes"]:>7,}  sigma {r["sigma"]:>7.3f}  '
          f'first bin {r["dk2"][0]:>11.3e}  SNR {r["snr"][0]:>7.2f}')

overlay(sweep, f'{V} — sensitivity to the wedge buffer')

## 5 · Keep a mask you like

**Save the arrays** — stage 5 loads `flag_mask.npy` when it exists, so this pins
the exact mask per variant, exclusion boxes included.

In [ ]:
for v in VARIANTS:
    path = cfgs[v].power_spectrum.flag_mask
    print(f'{v:<14s} -> {path}')
    # np.save(path, masks[v].astype(int))

**Or describe it in each variant config** — reproducible and self-documenting.
Delete `flag_mask.npy` first; stage 5 prefers the file.

```yaml
power_spectrum:
  mask:
    preset: paper          # TTGE III, arXiv:2604.24144
    wedge_buffer: null     # null = keep what the preset says
```

`preset` plus overrides covers the wedge, the rectangular limits and the streak
windows. Exclusion boxes are notebook-only — save the array if you need them.

Stage 5 prints the resolved preset, so the log records what was applied rather
than what was typed.